# XAI-ChestDiag - real experiment on Kaggle (NIH ChestX-ray14)

**Before running:**
1. *Add Input* -> **NIH Chest X-rays** (`nih-chest-xrays/data`).
2. *Settings* -> Accelerator -> **GPU T4 x2**; Internet **On**.
3. Run 1 trains `densenet121` + `efficientnet_b4`. Run 2 trains `vgg16` + `inception_v3`:
   add the output of Run 1 as an input (*Add Input -> Your Work*). Run 2 then reuses
   Run 1's trained models and resized images, and produces the full 4-model comparison.
4. **Save Version -> Save & Run All (Commit)**.


In [ ]:
MODELS = ['densenet121', 'efficientnet_b4']
SEEDS = [42]
RESIZE = 256

In [ ]:
!nvidia-smi
import torch, torchvision; print(torch.__version__, torchvision.__version__, torch.cuda.is_available())

## 1. Write the project code

In [ ]:
import os
os.makedirs('/kaggle/working/XAI-ChestDiag/src', exist_ok=True)
os.makedirs('/kaggle/working/XAI-ChestDiag/configs', exist_ok=True)
os.chdir('/kaggle/working/XAI-ChestDiag')

In [ ]:
%%writefile src/common.py
"""Shared helpers: configuration, seeding, device and image indexing."""
from __future__ import annotations

import os
import random
from pathlib import Path

import numpy as np
import yaml


def load_config(path: str | os.PathLike) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    import torch

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_device():
    import torch

    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def index_images(image_root: str | os.PathLike) -> dict[str, str]:
    """Map image file name -> full path (searches sub-folders recursively)."""
    root = Path(image_root)
    if not root.exists():
        raise FileNotFoundError(f"Image folder not found: {root}")
    index = {}
    for p in root.rglob("*.png"):
        index[p.name] = str(p)
    if not index:
        raise FileNotFoundError(f"No .png images found under {root}")
    return index


def ensure_dir(path: str | os.PathLike) -> Path:
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    return p


In [ ]:
%%writefile src/compare.py
"""Step 5 - Compare architectures on the same test images.

* Table of metrics (with 95% CIs) for every model
* Pairwise DeLong tests: proposed model vs. each baseline
* If several seeds were run, mean +/- SD across seeds

Usage:
    python -m src.compare --seed 42 --proposed densenet121
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd

from .common import ensure_dir, load_config
from .stats import delong_test


def fmt_ci(v, ci):
    return f"{v * 100:.1f} ({ci[0] * 100:.1f}-{ci[1] * 100:.1f})"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/default.yaml")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--proposed", default="densenet121")
    args = ap.parse_args()
    cfg = load_config(args.config)
    out_root = Path(cfg["output_dir"])
    summary = ensure_dir(out_root / "summary")

    thr = {}
    tp_file = summary / "throughput.json"
    if tp_file.exists():
        tp_rows = json.load(open(tp_file))
        bmax = max(x["batch_size"] for x in tp_rows)
        thr = {r["model"]: r["images_per_sec"] for r in tp_rows if r["batch_size"] == bmax}

    rows, preds = [], {}
    for name in cfg["models"]:
        d = out_root / name / f"seed{args.seed}"
        if not (d / "test_metrics.json").exists():
            print(f"[skip] {name}: no test_metrics.json in {d}")
            continue
        m = json.load(open(d / "test_metrics.json"))
        preds[name] = pd.read_csv(d / "test_predictions.csv")
        ci = m["ci95_bootstrap"]
        rows.append({
            "Model": name,
            "Parameters (M)": round(m["parameters"] / 1e6, 2),
            "Accuracy % (95% CI)": fmt_ci(m["accuracy"], ci["accuracy"]),
            "Precision %": fmt_ci(m["precision"], ci["precision"]),
            "Recall %": fmt_ci(m["sensitivity"], ci["sensitivity"]),
            "Specificity %": fmt_ci(m["specificity"], ci["specificity"]),
            "F1 %": fmt_ci(m["f1"], ci["f1"]),
            "AUC (95% CI, DeLong)": f"{m['auc']:.3f} ({m['auc_ci95_delong'][0]:.3f}-{m['auc_ci95_delong'][1]:.3f})",
            "Throughput (img/s)": thr.get(name, ""),
            "TP": m["TP"], "TN": m["TN"], "FP": m["FP"], "FN": m["FN"],
        })
    if not rows:
        raise SystemExit("No evaluated models found.")
    table = pd.DataFrame(rows)
    table.to_csv(summary / f"comparison_seed{args.seed}.csv", index=False)
    print(table.to_string(index=False))

    tests = []
    if args.proposed in preds:
        ref = preds[args.proposed].set_index("image")
        for name, p in preds.items():
            if name == args.proposed:
                continue
            p = p.set_index("image").loc[ref.index]
            assert (p.label.values == ref.label.values).all()
            t = delong_test(ref.label.values, ref.prob.values, p.prob.values)
            t.update(comparison=f"{args.proposed} vs {name}")
            tests.append(t)
        tdf = pd.DataFrame(tests)[["comparison", "auc_a", "auc_b", "auc_diff", "z", "p_value"]]
        tdf.to_csv(summary / f"delong_seed{args.seed}.csv", index=False)
        print("\nDeLong tests:\n" + tdf.to_string(index=False))

    # multi-seed aggregation (if available)
    agg = []
    for name in cfg["models"]:
        ms = [json.load(open(f)) for f in sorted((out_root / name).glob("seed*/test_metrics.json"))]
        if len(ms) > 1:
            r = {"Model": name, "n_seeds": len(ms)}
            for k in ["accuracy", "precision", "sensitivity", "specificity", "f1", "auc"]:
                v = np.array([x[k] for x in ms])
                r[k] = f"{v.mean():.4f} +/- {v.std(ddof=1):.4f}"
            agg.append(r)
    if agg:
        pd.DataFrame(agg).to_csv(summary / "multi_seed_mean_sd.csv", index=False)
        print("\nAcross seeds:\n" + pd.DataFrame(agg).to_string(index=False))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/dataset.py
"""PyTorch dataset and pre-processing / augmentation pipeline."""
from __future__ import annotations

import random

import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def read_gray(path: str) -> np.ndarray:
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise IOError(f"Cannot read image: {path}")
    return img


def minmax(img: np.ndarray) -> np.ndarray:
    """I_norm = (I - I_min) / (I_max - I_min)  -> [0, 1]."""
    img = img.astype(np.float32)
    lo, hi = float(img.min()), float(img.max())
    return (img - lo) / (hi - lo + 1e-8)


def to_tensor(img01: np.ndarray, imagenet_normalize: bool) -> torch.Tensor:
    x = np.repeat(img01[..., None], 3, axis=2)  # gray -> 3 channels
    if imagenet_normalize:
        x = (x - IMAGENET_MEAN) / IMAGENET_STD
    return torch.from_numpy(x.transpose(2, 0, 1).copy()).float()


def preprocess(path: str, size: int, imagenet_normalize: bool = True) -> tuple[torch.Tensor, np.ndarray]:
    """Deterministic evaluation pre-processing. Returns (tensor, resized uint8 image)."""
    img = read_gray(path)
    img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    return to_tensor(minmax(img), imagenet_normalize), img


class Augmenter:
    """Training-only augmentation: rotation, horizontal flip, zoom, CLAHE."""

    def __init__(self, cfg: dict):
        self.rot = cfg["rotation_deg"]
        self.hflip_p = cfg["hflip_p"]
        self.zoom = cfg["zoom"]
        self.clahe_p = cfg["clahe_p"]
        self.clahe = cv2.createCLAHE(clipLimit=cfg["clahe_clip_limit"],
                                     tileGridSize=(cfg["clahe_tile_grid"], cfg["clahe_tile_grid"]))

    def __call__(self, img: np.ndarray) -> np.ndarray:
        if random.random() < self.clahe_p:
            img = self.clahe.apply(img)
        h, w = img.shape
        angle = random.uniform(-self.rot, self.rot)
        scale = random.uniform(1 - self.zoom, 1 + self.zoom)
        M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, scale)
        img = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
        if random.random() < self.hflip_p:
            img = cv2.flip(img, 1)
        return img


class CXRDataset(Dataset):
    def __init__(self, df: pd.DataFrame, image_index: dict[str, str], size: int,
                 imagenet_normalize: bool = True, augment: Augmenter | None = None):
        missing = [im for im in df["image"] if im not in image_index]
        if missing:
            raise FileNotFoundError(f"{len(missing)} images listed in the CSV were not found, e.g. {missing[:3]}")
        self.paths = [image_index[im] for im in df["image"]]
        self.labels = df["label"].astype(np.float32).values
        self.size = size
        self.norm = imagenet_normalize
        self.augment = augment

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = read_gray(self.paths[i])
        img = cv2.resize(img, (self.size, self.size), interpolation=cv2.INTER_AREA)
        if self.augment is not None:
            img = self.augment(img)
        x = to_tensor(minmax(img), self.norm)
        return x, torch.tensor(self.labels[i])


In [ ]:
%%writefile src/evaluate.py
"""Step 3 - Evaluate a trained model on the held-out patient-level TEST split.

Writes:
    outputs/<model>/seed<seed>/test_predictions.csv   (one row per test image)
    outputs/<model>/seed<seed>/test_metrics.json      (metrics + 95% CIs)

Usage:
    python -m src.evaluate --model densenet121 --seed 42
"""
from __future__ import annotations

import argparse
import json

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from .common import get_device, index_images, load_config
from .dataset import CXRDataset
from .models import build_model, count_parameters
from .stats import binary_metrics, bootstrap_ci, delong_auc_ci


@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    probs = []
    for x, _ in loader:
        probs.append(torch.sigmoid(model(x.to(device))).float().cpu().numpy())
    return np.concatenate(probs)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/default.yaml")
    ap.add_argument("--model", default="densenet121")
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    cfg = load_config(args.config)
    pc, ec, tc = cfg["preprocessing"], cfg["evaluation"], cfg["training"]
    device = get_device()
    run_dir = f"{cfg['output_dir']}/{args.model}/seed{args.seed}"

    test_df = pd.read_csv(f"{cfg['data']['split_dir']}/test.csv")
    index = index_images(cfg["data"]["image_root"])
    ds = CXRDataset(test_df, index, pc["image_size"], pc["imagenet_normalize"])
    dl = DataLoader(ds, batch_size=tc["batch_size"], num_workers=tc["num_workers"], shuffle=False)

    model = build_model(args.model, pretrained=False, dropout=tc["dropout"]).to(device)
    model.load_state_dict(torch.load(f"{run_dir}/best.pt", map_location=device))
    prob = predict(model, dl, device)

    out = test_df[["image", "patient_id", "findings", "label"]].copy()
    out["prob"] = prob
    out.to_csv(f"{run_dir}/test_predictions.csv", index=False)

    y = out["label"].values
    m = binary_metrics(y, prob, ec["threshold"])
    auc, (lo, hi), se = delong_auc_ci(y, prob)
    m["auc_ci95_delong"] = [lo, hi]
    m["ci95_bootstrap"] = bootstrap_ci(y, prob, ec["threshold"], ec["bootstrap_iters"], seed=args.seed)
    m["n_test_images"] = int(len(y))
    m["n_test_abnormal"] = int(y.sum())
    m["n_test_normal"] = int((y == 0).sum())
    m["parameters"] = count_parameters(model)
    with open(f"{run_dir}/test_metrics.json", "w") as f:
        json.dump(m, f, indent=2)
    print(json.dumps(m, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/gradcam.py
"""Grad-CAM (Selvaraju et al., ICCV 2017).

    alpha_k^c = 1/Z * sum_i sum_j dY^c / dA_ij^k
    L^c       = ReLU( sum_k alpha_k^c A^k )

The heat-map is up-sampled to the input size and min-max normalised to [0, 1].
"""
from __future__ import annotations

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


class GradCAM:
    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model.eval()
        # in-place ReLUs would overwrite the stored activations / break hooks
        for m in self.model.modules():
            if isinstance(m, nn.ReLU):
                m.inplace = False
        self.acts = None
        self.grads = None
        self._h = target_layer.register_forward_hook(self._fwd)

    def _fwd(self, _m, _i, out):
        self.acts = out
        out.register_hook(self._save_grad)

    def _save_grad(self, g):
        self.grads = g

    def remove(self):
        self._h.remove()

    def __call__(self, x: torch.Tensor, target: str = "abnormal") -> tuple[np.ndarray, np.ndarray]:
        """x: (B,3,H,W). target='abnormal' explains the positive logit,
        'normal' explains the negative logit. Returns (cams [B,H,W] in [0,1], probs [B])."""
        self.model.zero_grad(set_to_none=True)
        logit = self.model(x)
        score = logit if target == "abnormal" else -logit
        score.sum().backward()
        weights = self.grads.mean(dim=(2, 3), keepdim=True)          # alpha_k^c
        cam = F.relu((weights * self.acts).sum(dim=1, keepdim=True))  # ReLU(sum_k alpha A^k)
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
        cam = cam.detach()
        b = cam.shape[0]
        flat = cam.view(b, -1)
        lo, hi = flat.min(1, keepdim=True)[0], flat.max(1, keepdim=True)[0]
        cam = ((flat - lo) / (hi - lo + 1e-8)).view_as(cam)
        return cam.cpu().numpy(), torch.sigmoid(logit).detach().cpu().numpy()


In [ ]:
%%writefile src/gradcam_figure.py
"""Step 8 - Grad-CAM qualitative figure from REAL test-set predictions.

Automatically selects, from the held-out test split, the most confidently
and correctly classified example for each requested finding plus one
correctly classified Normal case, and draws [radiograph | Grad-CAM overlay]
pairs. A CSV lists the exact images used, so the figure is reproducible.

Usage:
    python -m src.gradcam_figure --model densenet121 --seed 42 \
        --findings Pneumonia Infiltration Effusion
"""
from __future__ import annotations

import argparse

import cv2
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from .common import ensure_dir, get_device, index_images, load_config
from .dataset import preprocess
from .gradcam import GradCAM
from .models import build_model, gradcam_target_layer


def overlay(gray: np.ndarray, cam: np.ndarray, alpha=0.45) -> np.ndarray:
    heat = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)[..., ::-1] / 255.0
    base = np.repeat(gray[..., None] / 255.0, 3, axis=2)
    return np.clip((1 - alpha) * base + alpha * heat, 0, 1)


def pick_cases(pred: pd.DataFrame, findings: list[str], thr: float) -> list[tuple[str, str]]:
    chosen, used = [], set()
    for f in findings:
        cand = pred[(pred.label == 1) & (pred.prob >= thr) & pred.findings.str.contains(f, regex=False)]
        cand = cand[~cand.image.isin(used)].sort_values("prob", ascending=False)
        if len(cand):
            r = cand.iloc[0]
            chosen.append((r.image, f))
            used.add(r.image)
        else:
            print(f"[warn] no correctly classified test image with finding '{f}'")
    normal = pred[(pred.label == 0) & (pred.prob < thr)].sort_values("prob")
    if len(normal):
        chosen.append((normal.iloc[0].image, "No Finding"))
    return chosen


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/default.yaml")
    ap.add_argument("--model", default="densenet121")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--findings", nargs="+", default=["Pneumonia", "Infiltration", "Effusion"])
    ap.add_argument("--images", nargs="*", help="explicit image names instead of automatic selection")
    args = ap.parse_args()

    cfg = load_config(args.config)
    pc, tc, ec = cfg["preprocessing"], cfg["training"], cfg["evaluation"]
    device = get_device()
    run_dir = f"{cfg['output_dir']}/{args.model}/seed{args.seed}"
    pred = pd.read_csv(f"{run_dir}/test_predictions.csv")

    if args.images:
        cases = [(im, pred.set_index("image").loc[im, "findings"]) for im in args.images]
    else:
        cases = pick_cases(pred, args.findings, ec["threshold"])

    model = build_model(args.model, pretrained=False, dropout=tc["dropout"]).to(device)
    model.load_state_dict(torch.load(f"{run_dir}/best.pt", map_location=device))
    engine = GradCAM(model, gradcam_target_layer(model, args.model))
    index = index_images(cfg["data"]["image_root"])

    n = len(cases)
    fig, axes = plt.subplots(n, 2, figsize=(5.2, 2.7 * n))
    axes = np.atleast_2d(axes)
    records = []
    for i, (image, label) in enumerate(cases):
        x, gray = preprocess(index[image], pc["image_size"], pc["imagenet_normalize"])
        cam, prob = engine(x.unsqueeze(0).to(device), target="abnormal")
        axes[i, 0].imshow(gray, cmap="gray")
        im = axes[i, 1].imshow(overlay(gray, cam[0]))
        axes[i, 0].set_title(f"({chr(97 + i)}) {label}", fontsize=9, loc="left")
        axes[i, 1].set_title(f"Grad-CAM, P(abnormal) = {prob[0]:.3f}", fontsize=9, loc="left")
        for ax in axes[i]:
            ax.axis("off")
        records.append(dict(panel=chr(97 + i), image=image, findings=label, prob_abnormal=float(prob[0])))
    engine.remove()
    sm = plt.cm.ScalarMappable(cmap="jet", norm=plt.Normalize(0, 1))
    cbar = fig.colorbar(sm, ax=axes[:, 1].tolist(), fraction=0.04, pad=0.02)
    cbar.set_label("Grad-CAM activation (normalised)")

    out = ensure_dir(f"{cfg['output_dir']}/figures")
    fig.savefig(out / "fig_gradcam_cases.png", dpi=300, bbox_inches="tight")
    fig.savefig(out / "fig_gradcam_cases.pdf", bbox_inches="tight")
    pd.DataFrame(records).to_csv(out / "fig_gradcam_cases_images.csv", index=False)
    print(pd.DataFrame(records).to_string(index=False))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/models.py
"""Model zoo: DenseNet121 (proposed) and baselines VGG16, InceptionV3, EfficientNet-B4.

Every network ends with the same binary head:
    backbone features -> Global Average Pooling -> Dropout(p) -> Linear(1) -> (Sigmoid in the loss)
so that the comparison is fair. For VGG16 this means the original 3 fully
connected layers (~124 M parameters) are replaced by GAP + Linear, which keeps
the model at ~14.7 M parameters.
"""
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models


class BinaryHeadNet(nn.Module):
    def __init__(self, features: nn.Module, n_features: int, dropout: float, final_relu: bool = False):
        super().__init__()
        self.features = features
        self.final_relu = final_relu
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(n_features, 1)

    def forward(self, x):
        f = self.features(x)
        if self.final_relu:
            f = F.relu(f)
        f = torch.flatten(self.pool(f), 1)
        return self.fc(self.dropout(f)).squeeze(1)  # logits


def build_model(name: str, pretrained: bool = True, dropout: float = 0.3) -> nn.Module:
    name = name.lower()
    if name == "densenet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None)
        # torchvision applies ReLU after features.norm5 inside forward(); we keep that.
        return BinaryHeadNet(m.features, m.classifier.in_features, dropout, final_relu=True)
    if name == "vgg16":
        m = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1 if pretrained else None)
        return BinaryHeadNet(m.features, 512, dropout)
    if name == "efficientnet_b4":
        m = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.IMAGENET1K_V1 if pretrained else None)
        return BinaryHeadNet(m.features, m.classifier[1].in_features, dropout)
    if name == "inception_v3":
        m = models.inception_v3(weights=models.Inception_V3_Weights.IMAGENET1K_V1 if pretrained else None,
                                aux_logits=True, init_weights=not pretrained)
        m.transform_input = False  # we already normalise with ImageNet statistics
        feats = nn.Sequential(
            m.Conv2d_1a_3x3, m.Conv2d_2a_3x3, m.Conv2d_2b_3x3, m.maxpool1,
            m.Conv2d_3b_1x1, m.Conv2d_4a_3x3, m.maxpool2,
            m.Mixed_5b, m.Mixed_5c, m.Mixed_5d, m.Mixed_6a, m.Mixed_6b, m.Mixed_6c,
            m.Mixed_6d, m.Mixed_6e, m.Mixed_7a, m.Mixed_7b, m.Mixed_7c,
        )
        return BinaryHeadNet(feats, 2048, dropout)
    raise ValueError(f"Unknown model: {name}")


def gradcam_target_layer(model: nn.Module, name: str) -> nn.Module:
    """Last convolutional feature block used for Grad-CAM."""
    name = name.lower()
    if name == "densenet121":
        return model.features.denseblock4      # output of the final dense block (7x7x1024)
    if name == "vgg16":
        return model.features[28]              # last conv layer
    if name == "efficientnet_b4":
        return model.features[-1]
    if name == "inception_v3":
        return model.features[-1]              # Mixed_7c
    raise ValueError(name)


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


In [ ]:
%%writefile src/plots.py
"""Step 7 - Publication figures (300 dpi PNG + vector PDF), all from real outputs.

    fig_confusion_matrix   - proposed model, test set
    fig_roc_curves         - all models on the same test set, with AUCs
    fig_training_curves    - train / val loss and accuracy of the proposed model

Usage:
    python -m src.plots --seed 42 --proposed densenet121
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve

from .common import ensure_dir, load_config

NICE = {"densenet121": "DenseNet121 (proposed)", "vgg16": "VGG16",
        "inception_v3": "InceptionV3", "efficientnet_b4": "EfficientNet-B4"}
COLORS = {"densenet121": "#C0392B", "vgg16": "#7F8C8D", "inception_v3": "#2E86C1", "efficientnet_b4": "#28B463"}

plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "DejaVu Serif"],
                     "font.size": 11, "axes.spines.top": False, "axes.spines.right": False})


def save(fig, path: Path):
    fig.savefig(path.with_suffix(".png"), dpi=300, bbox_inches="tight")
    fig.savefig(path.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig)
    print("saved", path.with_suffix(".png"))


def confusion_matrix_fig(m: dict, path: Path):
    cm = np.array([[m["TN"], m["FP"]], [m["FN"], m["TP"]]])
    pct = cm / cm.sum(axis=1, keepdims=True) * 100
    fig, ax = plt.subplots(figsize=(4.6, 4))
    ax.imshow(pct, cmap="Blues", vmin=0, vmax=100)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]:,}\n({pct[i, j]:.1f}%)", ha="center", va="center",
                    color="white" if pct[i, j] > 50 else "black", fontsize=12)
    ax.set_xticks([0, 1], ["Normal", "Abnormal"])
    ax.set_yticks([0, 1], ["Normal", "Abnormal"])
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.spines[:].set_visible(False)
    ax.set_title(f"Test set (N = {cm.sum():,})", fontsize=11)
    save(fig, path)


def roc_fig(preds: dict[str, pd.DataFrame], metrics: dict[str, dict], path: Path):
    fig, ax = plt.subplots(figsize=(5, 4.6))
    for name, p in preds.items():
        fpr, tpr, _ = roc_curve(p.label, p.prob)
        lo, hi = metrics[name]["auc_ci95_delong"]
        ax.plot(fpr, tpr, lw=2 if name == "densenet121" else 1.4, color=COLORS.get(name),
                label=f"{NICE.get(name, name)}: AUC {metrics[name]['auc']:.3f} ({lo:.3f}-{hi:.3f})")
    ax.plot([0, 1], [0, 1], ls="--", color="#999", lw=1)
    ax.set_xlabel("False positive rate (1 - specificity)")
    ax.set_ylabel("True positive rate (sensitivity)")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.01)
    ax.legend(loc="lower right", fontsize=8.5, frameon=False)
    save(fig, path)


def training_fig(hist: pd.DataFrame, best_epoch: int, path: Path):
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
    for ax, key, lab in [(axes[0], "loss", "Binary cross-entropy loss"), (axes[1], "acc", "Accuracy")]:
        ax.plot(hist.epoch, hist[f"train_{key}"], label="Training", color="#2E86C1", lw=1.8)
        ax.plot(hist.epoch, hist[f"val_{key}"], label="Validation", color="#C0392B", lw=1.8)
        ax.axvline(best_epoch, ls=":", color="#555", lw=1)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(lab)
        ax.legend(frameon=False)
    axes[0].text(best_epoch, axes[0].get_ylim()[1], f" best epoch = {best_epoch}", va="top", fontsize=9)
    fig.tight_layout()
    save(fig, path)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/default.yaml")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--proposed", default="densenet121")
    args = ap.parse_args()
    cfg = load_config(args.config)
    root = Path(cfg["output_dir"])
    fig_dir = ensure_dir(root / "figures")

    preds, metrics = {}, {}
    for name in cfg["models"]:
        d = root / name / f"seed{args.seed}"
        if (d / "test_predictions.csv").exists():
            preds[name] = pd.read_csv(d / "test_predictions.csv")
            metrics[name] = json.load(open(d / "test_metrics.json"))
    if args.proposed in metrics:
        confusion_matrix_fig(metrics[args.proposed], fig_dir / "fig_confusion_matrix")
    if preds:
        roc_fig(preds, metrics, fig_dir / "fig_roc_curves")
    d = root / args.proposed / f"seed{args.seed}"
    if (d / "history.csv").exists():
        info = json.load(open(d / "train_info.json"))
        training_fig(pd.read_csv(d / "history.csv"), info["best_epoch"], fig_dir / "fig_training_curves")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/pointing_game.py
"""Step 6 - Quantitative explainability: Pointing Game (Zhang et al., ECCV 2016).

A "hit" is recorded when the maximum of the Grad-CAM heat-map falls inside a
radiologist bounding box (NIH BBox_List_2017.csv, coordinates in original
pixel space of the 1024x1024 PNGs), optionally with a tolerance margin (default 15 px, as in the
original Pointing Game protocol).

By default ONLY bounding-box images that belong to the held-out TEST patients
are used, so the explanation is evaluated on data the model never saw.
A trivial "image-centre" baseline is reported alongside for reference.

Usage:
    python -m src.pointing_game --model densenet121 --seed 42
"""
from __future__ import annotations

import argparse
import json

import numpy as np
import pandas as pd
import torch

from .common import ensure_dir, get_device, index_images, load_config
from .dataset import preprocess
from .gradcam import GradCAM
from .models import build_model, gradcam_target_layer


def load_bboxes(bbox_csv: str) -> pd.DataFrame:
    b = pd.read_csv(bbox_csv)
    b = b.iloc[:, :6]
    b.columns = ["image", "finding", "x", "y", "w", "h"]
    return b


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/default.yaml")
    ap.add_argument("--model", default="densenet121")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--tolerance", type=float, default=15.0, help="pixels, in original image space")
    ap.add_argument("--include-non-test", action="store_true",
                    help="also use bbox images of train/val patients (NOT recommended: leakage)")
    args = ap.parse_args()

    cfg = load_config(args.config)
    pc, tc = cfg["preprocessing"], cfg["training"]
    device = get_device()
    run_dir = f"{cfg['output_dir']}/{args.model}/seed{args.seed}"

    splits = pd.read_csv(f"{cfg['data']['split_dir']}/all_splits.csv")
    boxes = load_bboxes(cfg["data"]["bbox_csv"]).merge(splits[["image", "split"]], on="image", how="left")
    # NIH bounding boxes are given in the pixel space of the released 1024x1024 PNGs
    # (NOT the original scan size stored in Data_Entry_2017.csv).
    ref = float(cfg["data"].get("bbox_image_size", 1024))
    boxes["W"], boxes["H"] = ref, ref
    n_all = boxes.image.nunique()
    if not args.include_non_test:
        boxes = boxes[boxes.split == "test"]
    print(f"Bounding-box images: {n_all} in total, {boxes.image.nunique()} used "
          f"({len(boxes)} boxes).")
    if boxes.empty:
        raise SystemExit("No bounding-box images in the selected split.")

    model = build_model(args.model, pretrained=False, dropout=tc["dropout"]).to(device)
    model.load_state_dict(torch.load(f"{run_dir}/best.pt", map_location=device))
    cam_engine = GradCAM(model, gradcam_target_layer(model, args.model))
    index = index_images(cfg["data"]["image_root"])
    size = pc["image_size"]

    rows = []
    for image, grp in boxes.groupby("image"):
        x, _ = preprocess(index[image], size, pc["imagenet_normalize"])
        cam, prob = cam_engine(x.unsqueeze(0).to(device), target="abnormal")
        cy, cx = np.unravel_index(np.argmax(cam[0]), cam[0].shape)
        W, H = float(grp.W.iloc[0]), float(grp.H.iloc[0])
        px, py = (cx + 0.5) * W / size, (cy + 0.5) * H / size        # back to original pixels
        for _, b in grp.iterrows():
            t = args.tolerance
            hit = (b.x - t <= px <= b.x + b.w + t) and (b.y - t <= py <= b.y + b.h + t)
            centre_hit = (b.x - t <= W / 2 <= b.x + b.w + t) and (b.y - t <= H / 2 <= b.y + b.h + t)
            rows.append(dict(image=image, finding=b.finding, split=b.split, prob=float(prob[0]),
                             max_x=px, max_y=py, hit=int(hit), centre_baseline_hit=int(centre_hit)))
    cam_engine.remove()

    df = pd.DataFrame(rows)
    df.to_csv(f"{run_dir}/pointing_game_details.csv", index=False)
    per_class = df.groupby("finding").agg(boxes=("hit", "size"), hits=("hit", "sum"),
                                          accuracy=("hit", "mean"),
                                          centre_baseline=("centre_baseline_hit", "mean")).reset_index()
    per_class.to_csv(f"{run_dir}/pointing_game_per_class.csv", index=False)
    hits, n = int(df.hit.sum()), len(df)
    res = dict(model=args.model, seed=args.seed, tolerance_px=args.tolerance,
               images=int(df.image.nunique()), boxes=n, hits=hits, misses=n - hits,
               pointing_accuracy=hits / n, centre_baseline_accuracy=float(df.centre_baseline_hit.mean()),
               test_split_only=not args.include_non_test)
    with open(f"{run_dir}/pointing_game.json", "w") as f:
        json.dump(res, f, indent=2)
    print(per_class.to_string(index=False))
    print(json.dumps(res, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/prepare_splits.py
"""Step 1 - Build binary labels and a strict patient-level 70/10/20 split.

Label rule (emergency triage):
    "No Finding"            -> 0 (Normal)
    any of the 14 findings  -> 1 (Infected / Abnormal)

All images of a patient go to exactly one partition, so no patient appears
in more than one of train / val / test (no data leakage).

Usage:
    python -m src.prepare_splits --config configs/default.yaml
"""
from __future__ import annotations

import argparse
import json

import numpy as np
import pandas as pd

from .common import ensure_dir, load_config


def make_splits(labels_csv: str, train: float, val: float, test: float, seed: int) -> pd.DataFrame:
    assert abs(train + val + test - 1.0) < 1e-6, "split fractions must sum to 1"
    df = pd.read_csv(labels_csv)
    df = df.rename(columns={"Image Index": "image", "Finding Labels": "findings", "Patient ID": "patient_id"})
    df = df[["image", "findings", "patient_id", "View Position"]].copy()
    df["label"] = (df["findings"].str.strip() != "No Finding").astype(int)

    patients = df["patient_id"].unique()
    rng = np.random.default_rng(seed)
    rng.shuffle(patients)
    n = len(patients)
    n_train = int(round(train * n))
    n_val = int(round(val * n))
    part = {}
    for i, pid in enumerate(patients):
        part[pid] = "train" if i < n_train else ("val" if i < n_train + n_val else "test")
    df["split"] = df["patient_id"].map(part)
    return df


def summarize(df: pd.DataFrame) -> dict:
    out = {}
    for s in ["train", "val", "test"]:
        d = df[df.split == s]
        out[s] = {
            "images": int(len(d)),
            "patients": int(d.patient_id.nunique()),
            "abnormal": int(d.label.sum()),
            "normal": int((d.label == 0).sum()),
            "image_fraction": round(len(d) / len(df), 4),
        }
    out["total"] = {
        "images": int(len(df)),
        "patients": int(df.patient_id.nunique()),
        "abnormal": int(df.label.sum()),
        "normal": int((df.label == 0).sum()),
    }
    # leakage check
    sets = {s: set(df[df.split == s].patient_id) for s in ["train", "val", "test"]}
    overlap = len(sets["train"] & sets["val"]) + len(sets["train"] & sets["test"]) + len(sets["val"] & sets["test"])
    out["patient_overlap_between_splits"] = overlap
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/default.yaml")
    args = ap.parse_args()
    cfg = load_config(args.config)
    s = cfg["split"]
    df = make_splits(cfg["data"]["labels_csv"], s["train"], s["val"], s["test"], s["seed"])
    out_dir = ensure_dir(cfg["data"]["split_dir"])
    df.to_csv(out_dir / "all_splits.csv", index=False)
    for name in ["train", "val", "test"]:
        df[df.split == name].to_csv(out_dir / f"{name}.csv", index=False)
    stats = summarize(df)
    with open(out_dir / "split_summary.json", "w") as f:
        json.dump(stats, f, indent=2)
    print(json.dumps(stats, indent=2))
    assert stats["patient_overlap_between_splits"] == 0


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/resize_cache.py
"""Optional speed-up: pre-resize all NIH images once (1024x1024 -> 256x256).

Decoding 112k full-resolution PNGs every epoch is usually the bottleneck.
After running this, set `data.image_root` in the config to the output folder.
Bounding-box coordinates stay valid because the pointing game rescales using
the original width/height stored in Data_Entry_2017.csv.

Usage:
    python -m src.resize_cache --src data/nih/images --dst data/nih/images_256 --size 256
"""
from __future__ import annotations

import argparse
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import cv2


def _one(args):
    src, dst, size = args
    if Path(dst).exists():
        return
    img = cv2.imread(src, cv2.IMREAD_GRAYSCALE)
    cv2.imwrite(dst, cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--src", required=True)
    ap.add_argument("--dst", required=True)
    ap.add_argument("--size", type=int, default=256)
    ap.add_argument("--workers", type=int, default=8)
    a = ap.parse_args()
    dst = Path(a.dst)
    dst.mkdir(parents=True, exist_ok=True)
    jobs = [(str(p), str(dst / p.name), a.size) for p in Path(a.src).rglob("*.png")]
    print(f"{len(jobs)} images -> {dst}")
    with ProcessPoolExecutor(a.workers) as ex:
        for i, _ in enumerate(ex.map(_one, jobs, chunksize=256)):
            if i % 10000 == 0:
                print(i)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/stats.py
"""Statistics: classification metrics, DeLong's test (fast O(n log n) version,
Sun & Xu 2014) and bootstrap confidence intervals.

This module depends only on numpy / scipy / scikit-learn.
"""
from __future__ import annotations

import numpy as np
from scipy import stats as sps
from sklearn.metrics import roc_auc_score


# ---------------------------------------------------------------- metrics
def binary_metrics(y_true, y_prob, threshold: float = 0.5) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_pred = (np.asarray(y_prob) >= threshold).astype(int)
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    sens = tp / (tp + fn) if tp + fn else 0.0
    spec = tn / (tn + fp) if tn + fp else 0.0
    prec = tp / (tp + fp) if tp + fp else 0.0
    f1 = 2 * prec * sens / (prec + sens) if prec + sens else 0.0
    return dict(TP=tp, TN=tn, FP=fp, FN=fn,
                accuracy=(tp + tn) / len(y_true), sensitivity=sens, specificity=spec,
                precision=prec, f1=f1, auc=float(roc_auc_score(y_true, y_prob)), threshold=threshold)


# ---------------------------------------------------------------- DeLong
def _midrank(x: np.ndarray) -> np.ndarray:
    order = np.argsort(x)
    z = x[order]
    n = len(x)
    t = np.zeros(n, dtype=float)
    i = 0
    while i < n:
        j = i
        while j < n and z[j] == z[i]:
            j += 1
        t[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    out = np.empty(n, dtype=float)
    out[order] = t
    return out


def _fast_delong(preds_sorted: np.ndarray, m: int):
    """preds_sorted: (k, n) with the m positive examples first."""
    n = preds_sorted.shape[1] - m
    pos, neg = preds_sorted[:, :m], preds_sorted[:, m:]
    k = preds_sorted.shape[0]
    tx = np.array([_midrank(pos[r]) for r in range(k)])
    ty = np.array([_midrank(neg[r]) for r in range(k)])
    tz = np.array([_midrank(preds_sorted[r]) for r in range(k)])
    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    sx = np.atleast_2d(np.cov(v01))
    sy = np.atleast_2d(np.cov(v10))
    return aucs, sx / m + sy / n


def _prep(y_true, *probs):
    y = np.asarray(y_true).astype(int)
    order = np.argsort(-y, kind="mergesort")  # positives first
    m = int(y.sum())
    p = np.vstack([np.asarray(q, dtype=float)[order] for q in probs])
    return p, m


def delong_auc_ci(y_true, y_prob, alpha: float = 0.95):
    p, m = _prep(y_true, y_prob)
    aucs, cov = _fast_delong(p, m)
    auc, se = float(aucs[0]), float(np.sqrt(cov[0, 0]))
    z = sps.norm.ppf(1 - (1 - alpha) / 2)
    return auc, (max(0.0, auc - z * se), min(1.0, auc + z * se)), se


def delong_test(y_true, prob_a, prob_b) -> dict:
    """Two-sided DeLong test for two correlated ROC curves (same test cases)."""
    p, m = _prep(y_true, prob_a, prob_b)
    aucs, cov = _fast_delong(p, m)
    diff = aucs[0] - aucs[1]
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    z = diff / np.sqrt(var)
    pval = 2 * sps.norm.sf(abs(z))
    return dict(auc_a=float(aucs[0]), auc_b=float(aucs[1]), auc_diff=float(diff), z=float(z), p_value=float(pval))


# ---------------------------------------------------------------- bootstrap
def bootstrap_ci(y_true, y_prob, threshold=0.5, iters=2000, seed=0, alpha=0.95) -> dict:
    """Percentile bootstrap CIs for every metric in binary_metrics()."""
    y_true, y_prob = np.asarray(y_true), np.asarray(y_prob)
    rng = np.random.default_rng(seed)
    keys = ["accuracy", "sensitivity", "specificity", "precision", "f1", "auc"]
    samples = {k: [] for k in keys}
    n = len(y_true)
    for _ in range(iters):
        idx = rng.integers(0, n, n)
        if y_true[idx].min() == y_true[idx].max():
            continue
        m = binary_metrics(y_true[idx], y_prob[idx], threshold)
        for k in keys:
            samples[k].append(m[k])
    lo, hi = (1 - alpha) / 2 * 100, (1 + alpha) / 2 * 100
    return {k: (float(np.percentile(v, lo)), float(np.percentile(v, hi))) for k, v in samples.items()}


In [ ]:
%%writefile src/throughput.py
"""Step 4 - Measure inference throughput (images / second) of each architecture.

Model-only timing on random 224x224 tensors (no disk I/O), with warm-up and
CUDA synchronisation. Report the batch size together with the number.

Usage:
    python -m src.throughput --batch-sizes 1 64
"""
from __future__ import annotations

import argparse
import json
import time

import torch

from .common import ensure_dir, get_device, load_config
from .models import build_model, count_parameters


@torch.no_grad()
def measure(model, device, batch, size=224, warmup=20, iters=100, amp=False):
    model.eval()
    x = torch.randn(batch, 3, size, size, device=device)
    for _ in range(warmup):
        with torch.autocast(device_type=device.type, enabled=amp):
            model(x)
    if device.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        with torch.autocast(device_type=device.type, enabled=amp):
            model(x)
    if device.type == "cuda":
        torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    return batch * iters / dt, dt / iters * 1000 / batch


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/default.yaml")
    ap.add_argument("--batch-sizes", type=int, nargs="+", default=[1, 64])
    ap.add_argument("--iters", type=int, default=100)
    ap.add_argument("--amp", action="store_true", help="use mixed precision (FP16)")
    args = ap.parse_args()
    cfg = load_config(args.config)
    device = get_device()
    res = []
    for name in cfg["models"]:
        model = build_model(name, pretrained=False).to(device)
        for b in args.batch_sizes:
            ips, ms = measure(model, device, b, cfg["preprocessing"]["image_size"], iters=args.iters,
                              amp=args.amp and device.type == "cuda")
            row = dict(model=name, parameters_M=round(count_parameters(model) / 1e6, 2), batch_size=b,
                       images_per_sec=round(ips, 1), ms_per_image=round(ms, 3), device=str(device),
                       gpu=torch.cuda.get_device_name(0) if device.type == "cuda" else None, amp=args.amp)
            print(row)
            res.append(row)
        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()
    out = ensure_dir(f"{cfg['output_dir']}/summary")
    with open(out / "throughput.json", "w") as f:
        json.dump(res, f, indent=2)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/train.py
"""Step 2 - Train one model on the patient-level training split.

Usage:
    python -m src.train --model densenet121 --config configs/default.yaml --seed 42
"""
from __future__ import annotations

import argparse
import json
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from .common import ensure_dir, get_device, index_images, load_config, set_seed
from .dataset import Augmenter, CXRDataset
from .models import build_model, count_parameters


def run_epoch(model, loader, criterion, device, optimizer=None, scaler=None, amp=False):
    train = optimizer is not None
    model.train(train)
    total, n, correct = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, enabled=amp):
                logits = model(x)
                loss = criterion(logits.float(), y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total += loss.item() * x.size(0)
            n += x.size(0)
            correct += ((logits.detach() > 0).float() == y).sum().item()
    return total / n, correct / n


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/default.yaml")
    ap.add_argument("--model", default="densenet121")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--max-epochs", type=int, default=None, help="override (for quick tests)")
    args = ap.parse_args()

    cfg = load_config(args.config)
    tc, pc = cfg["training"], cfg["preprocessing"]
    set_seed(args.seed)
    device = get_device()
    amp = bool(tc["amp"]) and device.type == "cuda"
    out_dir = ensure_dir(f"{cfg['output_dir']}/{args.model}/seed{args.seed}")

    split_dir = cfg["data"]["split_dir"]
    train_df = pd.read_csv(f"{split_dir}/train.csv")
    val_df = pd.read_csv(f"{split_dir}/val.csv")
    index = index_images(cfg["data"]["image_root"])

    train_ds = CXRDataset(train_df, index, pc["image_size"], pc["imagenet_normalize"], Augmenter(cfg["augmentation"]))
    val_ds = CXRDataset(val_df, index, pc["image_size"], pc["imagenet_normalize"])
    g = torch.Generator().manual_seed(args.seed)
    kw = dict(batch_size=tc["batch_size"], num_workers=tc["num_workers"], pin_memory=device.type == "cuda")
    train_dl = DataLoader(train_ds, shuffle=True, drop_last=False, generator=g, **kw)
    val_dl = DataLoader(val_ds, shuffle=False, **kw)

    model = build_model(args.model, pretrained=tc["pretrained"], dropout=tc["dropout"]).to(device)
    n_params = count_parameters(model)
    print(f"{args.model}: {n_params / 1e6:.2f} M parameters | device={device} | amp={amp}")

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=tc["lr"], betas=tuple(tc["betas"]),
                                 weight_decay=tc["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min",
                                                           factor=tc["lr_factor"], patience=tc["lr_patience"])
    scaler = torch.amp.GradScaler(device.type, enabled=amp)

    epochs = args.max_epochs or tc["epochs"]
    best, best_epoch, wait, history = np.inf, -1, 0, []
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        tl, ta = run_epoch(model, train_dl, criterion, device, optimizer, scaler, amp)
        vl, va = run_epoch(model, val_dl, criterion, device, amp=amp)
        scheduler.step(vl)
        lr = optimizer.param_groups[0]["lr"]
        history.append(dict(epoch=epoch, train_loss=tl, train_acc=ta, val_loss=vl, val_acc=va, lr=lr))
        pd.DataFrame(history).to_csv(out_dir / "history.csv", index=False)
        print(f"epoch {epoch:03d} | train {tl:.4f}/{ta:.4f} | val {vl:.4f}/{va:.4f} | lr {lr:.2e}")
        if vl < best - 1e-6:
            best, best_epoch, wait = vl, epoch, 0
            torch.save(model.state_dict(), out_dir / "best.pt")
        else:
            wait += 1
            if wait >= tc["early_stopping_patience"]:
                print(f"Early stopping at epoch {epoch} (best epoch {best_epoch}).")
                break

    info = dict(model=args.model, seed=args.seed, parameters=n_params, best_epoch=best_epoch,
                best_val_loss=best, epochs_run=len(history), train_time_sec=time.time() - t0,
                device=str(device), gpu=torch.cuda.get_device_name(0) if device.type == "cuda" else None,
                torch_version=torch.__version__)
    with open(out_dir / "train_info.json", "w") as f:
        json.dump(info, f, indent=2)
    print(json.dumps(info, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile configs/default.yaml
# XAI-ChestDiag experiment configuration
# Paths -----------------------------------------------------------------
data:
  # Folder that contains the NIH ChestX-ray14 images (searched recursively,
  # so the original images_001 ... images_012 sub-folders are fine).
  image_root: data/nih/images
  labels_csv: data/nih/Data_Entry_2017.csv
  bbox_csv: data/nih/BBox_List_2017.csv
  bbox_image_size: 1024   # bbox coordinates refer to the 1024x1024 PNGs
  split_dir: outputs/splits

split:
  train: 0.70
  val: 0.10
  test: 0.20
  seed: 42

preprocessing:
  image_size: 224
  # After min-max scaling to [0, 1], standardise with ImageNet statistics
  # (required because the backbones are ImageNet-pretrained).
  imagenet_normalize: true

augmentation:          # applied to the training set only
  rotation_deg: 10
  hflip_p: 0.5
  zoom: 0.10
  clahe_p: 0.5
  clahe_clip_limit: 2.0
  clahe_tile_grid: 8

training:
  batch_size: 64
  epochs: 50
  lr: 1.0e-4
  betas: [0.9, 0.999]
  weight_decay: 0.0
  dropout: 0.3
  lr_factor: 0.5
  lr_patience: 3
  early_stopping_patience: 7
  num_workers: 8
  amp: true            # mixed precision on CUDA
  pretrained: true     # ImageNet weights

evaluation:
  threshold: 0.5       # decision threshold on P(abnormal)
  bootstrap_iters: 2000

models: [densenet121, vgg16, inception_v3, efficientnet_b4]

output_dir: outputs


In [ ]:
open('src/__init__.py','w').close()
!ls src

## 2. Dataset, image cache, config

In [ ]:
import glob, yaml, os, shutil
hits = glob.glob('/kaggle/input/**/Data_Entry_2017.csv', recursive=True)
assert hits, 'Add the NIH dataset (nih-chest-xrays/data) as notebook input'
NIH = os.path.dirname(sorted(hits, key=len)[0]); print('NIH dataset at', NIH)
bbox = glob.glob(os.path.join(NIH, '**/BBox_List_2017.csv'), recursive=True)[0]
# reuse resized images from a previous run if attached as input
prev = [d for d in glob.glob(f'/kaggle/input/**/images_{RESIZE}', recursive=True) if os.path.isdir(d)]
IMG = prev[0] if prev and len(os.listdir(prev[0])) == 112120 else f'/kaggle/working/images_{RESIZE}'
print('image cache:', IMG)
cfg = yaml.safe_load(open('configs/default.yaml'))
cfg['data'].update(image_root=IMG, labels_csv=os.path.join(NIH, 'Data_Entry_2017.csv'), bbox_csv=bbox,
                   split_dir='outputs/splits')
cfg['training']['num_workers'] = os.cpu_count()
cfg['models'] = ['densenet121', 'vgg16', 'inception_v3', 'efficientnet_b4']
yaml.safe_dump(cfg, open('configs/kaggle.yaml', 'w'), sort_keys=False)
print(yaml.safe_dump(cfg, sort_keys=False))

In [ ]:
# Sanity check (fails fast if anything is broken)
import sys, torch, pandas as pd
sys.path.insert(0, '.')
from src.models import build_model, gradcam_target_layer, count_parameters
from src.gradcam import GradCAM
from src.dataset import CXRDataset, Augmenter
from src.common import index_images
from src.pointing_game import load_bboxes
for m in MODELS:
    net = build_model(m, pretrained=True).cuda()
    x = torch.randn(2, 3, 224, 224).cuda()
    net(x).sum().backward()
    cam, p = GradCAM(net.eval(), gradcam_target_layer(net, m))(x[:1])
    print(f'{m:16s} params={count_parameters(net)/1e6:.2f}M cam={cam.shape} ok')
    del net; torch.cuda.empty_cache()
df = pd.read_csv(cfg['data']['labels_csv']).rename(columns={'Image Index': 'image'}).head(16)
df['label'] = (df['Finding Labels'] != 'No Finding').astype(int)
ds = CXRDataset(df, index_images(NIH), 224, True, Augmenter(cfg['augmentation']))
xb, yb = next(iter(torch.utils.data.DataLoader(ds, batch_size=8, num_workers=2)))
print('loader ok', xb.shape, yb.tolist())
print(load_bboxes(cfg['data']['bbox_csv']).head(2))


In [ ]:
if IMG.startswith('/kaggle/working'):
    !python -m src.resize_cache --src {NIH} --dst {IMG} --size {RESIZE} --workers {os.cpu_count()}
print(len(os.listdir(IMG)), 'images ready')

## 3. Import results of a previous run (if attached)

In [ ]:
for p in glob.glob('/kaggle/input/**/outputs/*/seed*/test_metrics.json', recursive=True):
    run_dir = os.path.dirname(p); model = run_dir.split('/')[-2]; seed = run_dir.split('/')[-1]
    dst = f'outputs/{model}/{seed}'
    if not os.path.exists(dst):
        shutil.copytree(run_dir, dst); print('imported', model, seed)

## 4. Patient-level split (identical in every run: fixed seed)

In [ ]:
!python -m src.prepare_splits --config configs/kaggle.yaml

## 5. Train + evaluate

In [ ]:
for seed in SEEDS:
    for m in MODELS:
        !python -m src.train --config configs/kaggle.yaml --model {m} --seed {seed}
        !python -m src.evaluate --config configs/kaggle.yaml --model {m} --seed {seed}

## 6. Throughput, comparison + DeLong, Pointing Game, figures

In [ ]:
!python -m src.throughput --config configs/kaggle.yaml --batch-sizes 1 64 --amp
!python -m src.compare --config configs/kaggle.yaml --seed {SEEDS[0]}
if os.path.exists(f'outputs/densenet121/seed{SEEDS[0]}/best.pt'):
    !python -m src.pointing_game --config configs/kaggle.yaml --model densenet121 --seed {SEEDS[0]}
    !python -m src.gradcam_figure --config configs/kaggle.yaml --model densenet121 --seed {SEEDS[0]} --findings Pneumonia Infiltration Effusion Consolidation
!python -m src.plots --config configs/kaggle.yaml --seed {SEEDS[0]}

## 7. Package results

In [ ]:
!cd outputs && zip -qr ../results.zip . -x '*.pt'
!ls -la results.zip
from IPython.display import Image, display
for f in sorted(glob.glob('outputs/figures/*.png')): print(f); display(Image(f, width=600))